# CreditWise Loan Approval Analysis

**Project:** CreditWise Loan System for SecureTrust Bank  
**Task:** Binary classification of historical loan approval outcomes (`Yes` / `No`).

## What this revised notebook does
- Audits the supplied 1,000-row dataset and preserves records with unknown target labels separately.
- Performs EDA with written interpretations to be completed from the actual plots and outputs.
- Checks duplicates, missingness, data types and potential outliers.
- Engineers interpretable financial ratios without using the target.
- Uses a stratified train/validation/test split and leakage-safe preprocessing pipelines.
- Compares a Dummy baseline with Logistic Regression, KNN and Gaussian Naive Bayes.
- Uses five-fold stratified cross-validation and hyperparameter tuning on training data only.
- Selects a model using validation F1, then evaluates the locked models on the test set.
- Examines errors, permutation importance and approval prediction differences by gender.

**Important:** The supplied CSV is retained as the raw source. Missing predictor values are imputed inside the training pipeline, not permanently filled in the source CSV. The 50 rows without `Loan_Approved` are not given invented labels and are excluded from supervised model training.

In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [2]:
# Load the original supplied dataset. Keep this file unchanged.
df_raw = pd.read_csv("loan_approval_data.csv")

print("Dataset shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())
df_raw.info()

Dataset shape: (1000, 20)
Columns: ['Applicant_ID', 'Applicant_Income', 'Coapplicant_Income', 'Employment_Status', 'Age', 'Marital_Status', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Gender', 'Employer_Category', 'Loan_Approved']


,Applicant_ID,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
0,1.0,17795.0,1387.0,Salaried,51.0,Married,0.0,637.0,4.0,0.53,19403.0,45638.0,16619.0,84.0,Personal,Urban,Not Graduate,Female,Private,No
1,2.0,2860.0,2679.0,Salaried,46.0,Married,3.0,621.0,2.0,0.30,2580.0,49272.0,38687.0,NaN,Car,Semiurban,Graduate,NaN,Private,No
2,3.0,7390.0,2106.0,Salaried,25.0,Single,2.0,674.0,4.0,0.20,13844.0,6908.0,27943.0,72.0,NaN,Urban,NaN,Female,Government,Yes
3,4.0,13964.0,8173.0,Salaried,40.0,Married,2.0,579.0,3.0,0.31,9553.0,10844.0,27819.0,60.0,Business,Rural,Graduate,Female,Government,No
4,5.0,13284.0,4223.0,Self-employed,31.0,Single,2.0,721.0,1.0,0.29,9386.0,37629.0,12741.0,72.0,Car,NaN,Graduate,Male,Private,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Applicant_ID        950 non-null    float64
 1   Applicant_Income    950 non-null    float64
 2   Coapplicant_Income  950 non-null    float64
 3   Employment_Status   950 non-null    object 
 4   Age                 950 non-null    float64
 5   Marital_Status      950 non-null    object 
 6   Dependents          950 non-null    float64
 7   Credit_Score        950 non-null    float64
 8   Existing_Loans      950 non-null    float64
 9   DTI_Ratio           950 non-null    float64
 10  Savings             950 non-null    float64
 11  Collateral_Value    950 non-null    float64
 12  Loan_Amount         950 non-null    float64
 13  Loan_Term           950 non-null    float64
 14  Loan_Purpose        950 non-null    object 
 15  Property_Area       950 non-null    object 
 16  Educati

## 1. Data quality audit

First inspect missing values, duplicate records, data types, target labels and numerical ranges. Do not impute the target. `Applicant_ID` is an identifier and will not be used as a model predictor.

In [3]:
print("Missing values by column:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Non-missing unique Applicant_ID values:", df_raw["Applicant_ID"].nunique())
print("Non-missing Applicant_ID count:", df_raw["Applicant_ID"].notna().sum())

print("Target counts, including missing labels:")
display(df_raw["Loan_Approved"].value_counts(dropna=False).to_frame("count"))

print("Numerical summary:")
display(df_raw.describe().T)

Missing values by column:


,missing_count
Applicant_ID,50
Applicant_Income,50
Employer_Category,50
Gender,50
Education_Level,50
Property_Area,50
Loan_Purpose,50
Loan_Term,50
Loan_Amount,50
Collateral_Value,50


Exact duplicate rows: 0
Non-missing unique Applicant_ID values: 950
Non-missing Applicant_ID count: 950
Target counts, including missing labels:


,count
Loan_Approved,
No,652
Yes,298
NaN,50


Numerical summary:


,count,mean,std,min,25%,50%,75%,max
Applicant_ID,950.0,501.220000,289.608451,1.0,250.25,499.50,752.75,1000.0
Applicant_Income,950.0,10852.571579,5061.632859,2009.0,6730.75,10548.00,15190.00,19988.0
Coapplicant_Income,950.0,5082.455789,2943.161570,1.0,2472.75,5205.50,7620.75,9996.0
Age,950.0,39.971579,11.139797,21.0,30.25,40.00,49.00,59.0
Dependents,950.0,1.474737,1.105067,0.0,1.00,1.00,2.00,3.0
Credit_Score,950.0,676.033684,71.346015,550.0,616.25,678.00,737.00,799.0
Existing_Loans,950.0,1.950526,1.406246,0.0,1.00,2.00,3.00,4.0
DTI_Ratio,950.0,0.347263,0.144341,0.1,0.22,0.34,0.48,0.6
Savings,950.0,9940.452632,5860.736885,65.0,4760.25,9880.50,15074.50,19996.0
Collateral_Value,950.0,24802.792632,14345.696031,36.0,12698.25,24321.00,36947.00,49954.0


### Target handling and dataset partitions

Rows with unknown approval outcomes cannot be used for supervised training or evaluation. Keep them in a separate file for transparency. Do not fill missing `Loan_Approved` values with the majority class because that would invent labels.

In [4]:
# Remove only exact duplicate rows, if any. The supplied file currently has none.
df = df_raw.drop_duplicates().copy()

labelled_df = df[df["Loan_Approved"].notna()].copy()
unlabelled_df = df[df["Loan_Approved"].isna()].copy()

# Normalise label text and map it to 0/1.
labelled_df["Loan_Approved"] = labelled_df["Loan_Approved"].astype(str).str.strip().str.title()
label_map = {"No": 0, "Yes": 1}
labelled_df["Loan_Approved"] = labelled_df["Loan_Approved"].map(label_map)

if labelled_df["Loan_Approved"].isna().any():
    raise ValueError("Unexpected target labels found. Inspect the source values before modelling.")

labelled_df["Loan_Approved"] = labelled_df["Loan_Approved"].astype(int)

labelled_df.to_csv("loan_approval_labeled.csv", index=False)
unlabelled_df.to_csv("loan_approval_unlabeled.csv", index=False)

print("Labelled rows:", len(labelled_df))
print("Unlabelled rows:", len(unlabelled_df))
print("Approval rate among labelled rows:", labelled_df["Loan_Approved"].mean().round(3))
print("Target distribution:")
display(labelled_df["Loan_Approved"].value_counts().rename(index={0:"No", 1:"Yes"}).to_frame("count"))

Labelled rows: 950
Unlabelled rows: 50
Approval rate among labelled rows: 0.314
Target distribution:


,count
Loan_Approved,
No,652
Yes,298


## 2. Exploratory Data Analysis (EDA)

The following plots use labelled records for approval comparisons. Write interpretations from the observed plots; do not claim that a relationship is causal.

In [5]:
# Target distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=labelled_df, x="Loan_Approved", color="steelblue")
ax.set_xticks([0, 1], ["Rejected (No)", "Approved (Yes)"])
ax.set_title("Loan approval class distribution")
ax.set_xlabel("Loan outcome")
ax.set_ylabel("Number of applicants")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

print("Class proportions:")
display(labelled_df["Loan_Approved"].value_counts(normalize=True).rename(index={0:"No",1:"Yes"}).mul(100).round(1).to_frame("percentage"))

Class proportions:


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/3453462077.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,percentage
Loan_Approved,
No,68.6
Yes,31.4


In [6]:
# Distributions of important numeric variables
numeric_eda = ["Applicant_Income", "Coapplicant_Income", "Age", "Credit_Score",
               "DTI_Ratio", "Savings", "Collateral_Value", "Loan_Amount"]
labelled_df[numeric_eda].hist(figsize=(14, 10), bins=20, edgecolor="black")
plt.suptitle("Distributions of applicant and loan characteristics", y=1.02)
plt.tight_layout()
plt.show()

/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/3954673488.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Numeric variables by approval outcome
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flat, ["Credit_Score", "DTI_Ratio", "Applicant_Income", "Loan_Amount"]):
    sns.boxplot(data=labelled_df, x="Loan_Approved", y=col, ax=ax)
    ax.set_xticks([0, 1], ["No", "Yes"])
    ax.set_title(f"{col} by loan outcome")
plt.tight_layout()
plt.show()

/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/1964845518.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Approval rates across categorical groups
categorical_eda = ["Employment_Status", "Marital_Status", "Loan_Purpose",
                   "Property_Area", "Education_Level", "Employer_Category", "Gender"]
for col in categorical_eda:
    rate = labelled_df.groupby(col, dropna=False)["Loan_Approved"].mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    ax = sns.barplot(x=rate.index.astype(str), y=rate.values, color="steelblue")
    ax.set_title(f"Approval rate by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Approval rate")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
    display(rate.rename("approval_rate").to_frame())

/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2112034784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approval_rate
Employment_Status,
Contract,0.396985
Self-employed,0.312139
NaN,0.295455
Salaried,0.292411
Unemployed,0.244186


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2112034784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approval_rate
Marital_Status,
Single,0.337313
NaN,0.312500
Married,0.299824


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2112034784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approval_rate
Loan_Purpose,
NaN,0.437500
Personal,0.349112
Business,0.317708
Home,0.316667
Education,0.301775
Car,0.255208


... (4 outputs left out)

/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2112034784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approval_rate
Employer_Category,
MNC,0.397059
Private,0.330484
Business,0.307692
Government,0.277487
Unemployed,0.273684
NaN,0.191489


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2112034784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approval_rate
Gender,
NaN,0.382979
Female,0.342466
Male,0.288104


### EDA interpretation prompts

For each plot, record (1) what is visibly different, (2) the numerical evidence supporting the observation, and (3) one limitation. For example, an approval-rate difference across employment categories is an association in this dataset, not proof that employment category alone causes approval.

### Initial findings from the supplied data

The following are descriptive findings calculated from the 950 rows with known approval outcomes. Recheck these values if the source CSV changes.

- **Class distribution:** 298 applications are marked approved and 652 rejected. The approval rate is approximately **31.4%**, so accuracy alone could be misleading.
- **Credit score:** The mean credit score is approximately **726.9** for approved applications and **651.6** for rejected applications. This is a clear association in this dataset, not proof of causation.
- **Debt-to-income ratio:** The mean DTI is approximately **0.25** for approved applications and **0.39** for rejected applications. Lower DTI is associated with approval in these records.
- **Applicant income:** Mean applicant income is approximately **11,797** among approved records and **10,406** among rejected records.
- **Loan amount:** Mean requested loan amount is approximately **18,240** among approved records and **21,473** among rejected records.

These patterns should be discussed alongside the model results and the dataset's limitations. Do not interpret them as bank policy or causal effects.

### Dataset-description consistency check

The project PDF lists a smaller set of categories than the CSV actually contains. For example, `Employment_Status` in the CSV includes `Contract` and `Unemployed`, while the PDF describes `Salaried / Self-Employed / Business`. The CSV's `Employer_Category` includes `Government`, `Private`, `MNC`, `Business` and `Unemployed`, which also differs from the abbreviated PDF description. Use the values actually present in the CSV for analysis, and explicitly mention this documentation mismatch in the report rather than silently changing category names.

In [9]:
# Inspect the actual categories supplied in the CSV; this is the data dictionary evidence.
for col in categorical_eda:
    print(f"\n{col}: {sorted(labelled_df[col].dropna().astype(str).unique().tolist())}")


Employment_Status: ['Contract', 'Salaried', 'Self-employed', 'Unemployed']

Marital_Status: ['Married', 'Single']

Loan_Purpose: ['Business', 'Car', 'Education', 'Home', 'Personal']

Property_Area: ['Rural', 'Semiurban', 'Urban']

Education_Level: ['Graduate', 'Not Graduate']

Employer_Category: ['Business', 'Government', 'MNC', 'Private', 'Unemployed']

Gender: ['Female', 'Male']


## 3. Outlier and duplicate analysis

IQR flags are a screening tool, not an automatic deletion rule. Unusually high or low financial values may be legitimate applicants. Investigate flagged records against plausible domain ranges before changing them. This project retains valid observations and uses a robust median imputer; no rows are removed solely for being IQR outliers.

In [10]:
continuous_cols = ["Applicant_Income", "Coapplicant_Income", "Age", "Credit_Score",
                   "DTI_Ratio", "Savings", "Collateral_Value", "Loan_Amount"]
outlier_rows = []
for col in continuous_cols:
    s = labelled_df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append({"feature": col, "lower_IQR_bound": lower, "upper_IQR_bound": upper,
                         "flagged_count": count, "flagged_percent": 100 * count / len(s)})
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary.round(2))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, continuous_cols):
    sns.boxplot(x=labelled_df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

,feature,lower_IQR_bound,upper_IQR_bound,flagged_count,flagged_percent
0,Applicant_Income,-5906.38,27822.62,0,0.0
1,Coapplicant_Income,-5210.25,15269.75,0,0.0
2,Age,4.00,76.00,0,0.0
3,Credit_Score,437.50,913.50,0,0.0
4,DTI_Ratio,-0.17,0.87,0,0.0
5,Savings,-10675.50,30488.50,0,0.0
6,Collateral_Value,-23679.50,73156.50,0,0.0
7,Loan_Amount,-20776.50,60687.50,0,0.0


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/1833467354.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Feature engineering

Create a small number of interpretable financial features. These are deterministic transformations of applicant inputs and do not use the approval target. Keep the original features as well, so the model can use both raw and derived information.

In [11]:
model_df = labelled_df.copy()

model_df["Total_Income"] = model_df["Applicant_Income"] + model_df["Coapplicant_Income"]
model_df["Loan_to_Income_Ratio"] = model_df["Loan_Amount"] / model_df["Total_Income"].replace(0, np.nan)
model_df["Collateral_to_Loan_Ratio"] = model_df["Collateral_Value"] / model_df["Loan_Amount"].replace(0, np.nan)

print("Added features: Total_Income, Loan_to_Income_Ratio, Collateral_to_Loan_Ratio")
display(model_df[["Applicant_Income", "Coapplicant_Income", "Total_Income", "Loan_Amount",
                  "Loan_to_Income_Ratio", "Collateral_Value", "Collateral_to_Loan_Ratio"]].head())

Added features: Total_Income, Loan_to_Income_Ratio, Collateral_to_Loan_Ratio


,Applicant_Income,Coapplicant_Income,Total_Income,Loan_Amount,Loan_to_Income_Ratio,Collateral_Value,Collateral_to_Loan_Ratio
0,17795.0,1387.0,19182.0,16619.0,0.866385,45638.0,2.746134
1,2860.0,2679.0,5539.0,38687.0,6.984474,49272.0,1.273606
2,7390.0,2106.0,9496.0,27943.0,2.942607,6908.0,0.247218
3,13964.0,8173.0,22137.0,27819.0,1.256674,10844.0,0.389806
4,13284.0,4223.0,17507.0,12741.0,0.727766,37629.0,2.953379


## 5. Train / validation / test split

Use a stratified 70% / 15% / 15% split. Stratification preserves the approximate class proportions in each subset. The test set is held back until model selection and tuning are complete.

`Applicant_ID` is excluded because it is an identifier, not a meaningful applicant characteristic. `Gender` is also excluded from model predictors and retained only for a later subgroup fairness check. This avoids directly using gender to make predictions, although other variables may still act as proxies.

In [12]:
TARGET = "Loan_Approved"
DROP_FROM_FEATURES = ["Applicant_ID", TARGET, "Gender"]
X = model_df.drop(columns=DROP_FROM_FEATURES)
y = model_df[TARGET].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test.shape)
for name, target_part in [("Train", y_train), ("Validation", y_valid), ("Test", y_test)]:
    print(f"{name} approval proportion: {target_part.mean():.3f}; class counts: {target_part.value_counts().to_dict()}")

Train: (665, 20) Validation: (142, 20) Test: (143, 20)
Train approval proportion: 0.314; class counts: {0: 456, 1: 209}
Validation approval proportion: 0.310; class counts: {0: 98, 1: 44}
Test approval proportion: 0.315; class counts: {0: 98, 1: 45}


## 6. Leakage-safe preprocessing

All learned preprocessing happens inside a scikit-learn pipeline. Median imputation and scaling are fitted only on the training fold during cross-validation. Categorical missing values are imputed using the most frequent training-fold category, then one-hot encoded. Unknown categories at validation/test time are safely ignored.

In [13]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['Applicant_Income', 'Coapplicant_Income', 'Age', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term', 'Total_Income', 'Loan_to_Income_Ratio', 'Collateral_to_Loan_Ratio']
Categorical features: ['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Employer_Category']


## 7. Baseline and model training

The Dummy Classifier is the baseline. Compare it with the three algorithms already present in the original project: Logistic Regression, KNN and Gaussian Naive Bayes. Tune hyperparameters using five-fold stratified cross-validation on the training set only. F1 is the tuning metric because both false approvals and missed approvals matter; precision, recall and ROC-AUC are also reported.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])
baseline.fit(X_train, y_train)

model_specs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1, 10], "model__class_weight": [None, "balanced"]}
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [3, 5, 7, 9, 11],
         "model__weights": ["uniform", "distance"],
         "model__p": [1, 2]}
    ),
    "Gaussian Naive Bayes": (
        GaussianNB(),
        {"model__var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5]}
    )
}

tuned_models = {}
cv_rows = []
for model_name, (estimator, param_grid) in model_specs.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    search = GridSearchCV(
        estimator=pipe, param_grid=param_grid, scoring="f1", cv=cv,
        n_jobs=-1, refit=True, return_train_score=True
    )
    search.fit(X_train, y_train)
    tuned_models[model_name] = search.best_estimator_
    cv_rows.append({"model": model_name, "best_cv_f1": search.best_score_, "best_params": search.best_params_})
    print(f"{model_name}: best CV F1 = {search.best_score_:.3f}")
    print("Best parameters:", search.best_params_)

cv_results = pd.DataFrame(cv_rows).sort_values("best_cv_f1", ascending=False)
display(cv_results)

Logistic Regression: best CV F1 = 0.789
Best parameters: {'model__C': 0.1, 'model__class_weight': 'balanced'}


KNN: best CV F1 = 0.747
Best parameters: {'model__n_neighbors': 9, 'model__p': 2, 'model__weights': 'uniform'}
Gaussian Naive Bayes: best CV F1 = 0.748
Best parameters: {'model__var_smoothing': 1e-11}


,model,best_cv_f1,best_params
0,Logistic Regression,0.789022,"{'model__C': 0.1, 'model__class_weight': 'bala..."
2,Gaussian Naive Bayes,0.747745,{'model__var_smoothing': 1e-11}
1,KNN,0.746641,"{'model__n_neighbors': 9, 'model__p': 2, 'mode..."


## 8. Validation comparison and model selection

Use the validation set to choose the model. The test set must not be used to select the winner or tune parameters.

In [15]:
def calculate_metrics(model, X_part, y_part):
    pred = model.predict(X_part)
    result = {
        "accuracy": accuracy_score(y_part, pred),
        "precision": precision_score(y_part, pred, zero_division=0),
        "recall": recall_score(y_part, pred, zero_division=0),
        "f1": f1_score(y_part, pred, zero_division=0)
    }
    if hasattr(model, "predict_proba"):
        result["roc_auc"] = roc_auc_score(y_part, model.predict_proba(X_part)[:, 1])
    else:
        result["roc_auc"] = np.nan
    return result

validation_models = {"Dummy baseline": baseline, **tuned_models}
validation_rows = []
for name, model in validation_models.items():
    validation_rows.append({"model": name, **calculate_metrics(model, X_valid, y_valid)})

validation_results = pd.DataFrame(validation_rows).sort_values("f1", ascending=False)
display(validation_results.round(3))

# The validation F1 is the pre-declared selection criterion.
best_model_name = validation_results.iloc[0]["model"]
print("Selected using validation F1:", best_model_name)

,model,accuracy,precision,recall,f1,roc_auc
3,Gaussian Naive Bayes,0.894,0.854,0.795,0.824,0.942
1,Logistic Regression,0.831,0.685,0.841,0.755,0.922
2,KNN,0.845,0.789,0.682,0.732,0.909
0,Dummy baseline,0.690,0.000,0.000,0.000,0.500


Selected using validation F1: Gaussian Naive Bayes


## 9. Final test evaluation

After selecting the model using validation results, refit the tuned models on the combined training + validation data and evaluate them on the untouched test set. The test metrics are the final generalisation estimates. Do not change the model or its settings based on these test results.

In [16]:
X_trainval = pd.concat([X_train, X_valid], axis=0)
y_trainval = pd.concat([y_train, y_valid], axis=0)

final_models = {}
# Refit baseline and all tuned model configurations on train + validation.
final_baseline = baseline
final_baseline.fit(X_trainval, y_trainval)
final_models["Dummy baseline"] = final_baseline

for name, model in tuned_models.items():
    model.fit(X_trainval, y_trainval)
    final_models[name] = model

final_rows = []
for name, model in final_models.items():
    final_rows.append({"model": name, **calculate_metrics(model, X_test, y_test)})

final_results = pd.DataFrame(final_rows).sort_values("f1", ascending=False)
display(final_results.round(3))

selected_model = final_models[best_model_name]
print("Final selected model (selection based on validation):", best_model_name)

,model,accuracy,precision,recall,f1,roc_auc
1,Logistic Regression,0.860,0.727,0.889,0.800,0.950
2,KNN,0.860,0.805,0.733,0.767,0.907
3,Gaussian Naive Bayes,0.853,0.786,0.733,0.759,0.941
0,Dummy baseline,0.685,0.000,0.000,0.000,0.500


Final selected model (selection based on validation): Gaussian Naive Bayes


## 10. Error analysis of the selected model

A false positive is a rejected-in-history applicant predicted as approved (`actual=0`, `predicted=1`). A false negative is an approved-in-history applicant predicted as rejected (`actual=1`, `predicted=0`). Their business costs may differ, so discuss both rather than reporting accuracy alone.

In [17]:
y_test_pred = selected_model.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["Rejected (No)", "Approved (Yes)"], zero_division=0))

cm = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()

error_analysis = X_test.copy()
error_analysis["actual"] = y_test
error_analysis["predicted"] = y_test_pred
error_analysis["error_type"] = np.select(
    [(error_analysis.actual == 0) & (error_analysis.predicted == 1),
     (error_analysis.actual == 1) & (error_analysis.predicted == 0)],
    ["False positive", "False negative"], default="Correct"
)
print("Error counts:")
display(error_analysis["error_type"].value_counts().to_frame("count"))
display(error_analysis[error_analysis["error_type"] != "Correct"].head(10))

                precision    recall  f1-score   support

 Rejected (No)       0.88      0.91      0.89        98
Approved (Yes)       0.79      0.73      0.76        45

      accuracy                           0.85       143
     macro avg       0.83      0.82      0.83       143
  weighted avg       0.85      0.85      0.85       143

Error counts:


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/1777518828.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,count
error_type,
Correct,122
False negative,12
False positive,9


,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,...,Loan_Purpose,Property_Area,Education_Level,Employer_Category,Total_Income,Loan_to_Income_Ratio,Collateral_to_Loan_Ratio,actual,predicted,error_type
721,8767.0,6936.0,Contract,35.0,Single,3.0,641.0,1.0,0.18,11953.0,...,Home,Rural,Graduate,Business,15703.0,1.591097,1.295017,0,1,False positive
710,4472.0,8234.0,Contract,50.0,Single,0.0,781.0,2.0,0.37,16391.0,...,Car,Urban,Graduate,Unemployed,12706.0,1.135605,1.105898,1,0,False negative
664,10150.0,6924.0,Salaried,57.0,Single,1.0,669.0,1.0,0.39,9884.0,...,Education,Rural,Graduate,Private,17074.0,0.675179,1.536954,1,0,False negative
885,5072.0,9917.0,Salaried,50.0,NaN,1.0,728.0,4.0,0.34,7876.0,...,Home,Urban,Graduate,Business,14989.0,2.076256,0.098326,0,1,False positive
436,15923.0,9768.0,Contract,25.0,Single,1.0,702.0,0.0,0.40,5137.0,...,Business,Urban,NaN,Government,25691.0,1.327002,0.309867,0,1,False positive
464,13493.0,634.0,Salaried,57.0,NaN,1.0,684.0,4.0,0.20,2328.0,...,Business,Rural,Not Graduate,Unemployed,14127.0,1.528492,1.228129,1,0,False negative
304,12966.0,3741.0,Salaried,53.0,Married,2.0,636.0,4.0,0.14,15732.0,...,Home,Rural,Graduate,Business,16707.0,1.765128,1.506952,0,1,False positive
779,12161.0,7000.0,Salaried,34.0,Married,1.0,709.0,3.0,0.34,16978.0,...,Education,Semiurban,Graduate,Unemployed,19161.0,0.467877,3.840825,1,0,False negative
860,9783.0,6099.0,Contract,31.0,Single,0.0,703.0,1.0,NaN,2575.0,...,Education,Rural,Not Graduate,Private,15882.0,1.650170,1.844780,0,1,False positive
587,12176.0,9847.0,Salaried,46.0,Single,0.0,753.0,3.0,0.41,NaN,...,Personal,Rural,Graduate,Private,22023.0,0.773646,1.749149,0,1,False positive


## 11. ROC curve comparison

In [18]:
plt.figure(figsize=(8, 6))
for name, model in final_models.items():
    if hasattr(model, "predict_proba"):
        RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.title("ROC curves on the held-out test set")
plt.tight_layout()
plt.show()

/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/2042446131.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Feature importance

Permutation importance estimates how much the selected model's F1 changes when one input column is shuffled. It is model-agnostic, but correlated features can share or mask importance. Treat it as a diagnostic, not proof of causation.

In [19]:
perm = permutation_importance(
    selected_model, X_test, y_test, n_repeats=20,
    random_state=RANDOM_STATE, scoring="f1", n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance.head(12), x="importance_mean", y="feature", color="steelblue")
plt.title("Top permutation importances")
plt.xlabel("Mean decrease in F1 after shuffling")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

,feature,importance_mean,importance_std
6,Credit_Score,0.257198,0.035244
8,DTI_Ratio,0.242962,0.063233
18,Loan_to_Income_Ratio,0.044175,0.031302
19,Collateral_to_Loan_Ratio,0.027850,0.011118
11,Loan_Amount,0.003343,0.010566
5,Dependents,0.000000,0.000000
9,Savings,0.000000,0.000000
1,Coapplicant_Income,0.000000,0.000000
3,Age,-0.000919,0.006704
14,Property_Area,-0.004937,0.006728


/var/folders/cl/x8xwpzds4b12ymc8qb0bv88r0000gn/T/ipykernel_6554/4270253495.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Fairness / subgroup diagnostic

Gender is excluded from model predictors but retained in the held-out records for a descriptive subgroup check. Compare subgroup sample sizes, approval prevalence, prediction rates and recall. Small subgroup counts make estimates unstable. This check does not certify the model as unbiased; age and other attributes may still act as proxies, and the dataset's provenance and representativeness are unknown.

In [20]:
# Recover gender values for the same held-out rows using their original index.
gender_test = labelled_df.loc[X_test.index, "Gender"]
fairness_df = pd.DataFrame({
    "Gender": gender_test,
    "actual": y_test,
    "predicted": y_test_pred
})

group_rows = []
for group, part in fairness_df.groupby("Gender", dropna=False):
    group_rows.append({
        "group": str(group),
        "n": len(part),
        "actual_approval_rate": part["actual"].mean(),
        "predicted_approval_rate": part["predicted"].mean(),
        "recall_for_approved": recall_score(part["actual"], part["predicted"], zero_division=0),
        "precision_for_approved": precision_score(part["actual"], part["predicted"], zero_division=0)
    })

group_metrics = pd.DataFrame(group_rows)
display(group_metrics.round(3))

,group,n,actual_approval_rate,predicted_approval_rate,recall_for_approved,precision_for_approved
0,Female,57,0.386,0.333,0.682,0.789
1,Male,82,0.280,0.280,0.783,0.783
2,nan,4,0.000,0.000,0.000,0.000


## 14. Save reproducible outputs

The selected fitted pipeline includes preprocessing and the classifier, so future predictions use the same imputation, encoding and scaling steps. Keep the original CSV and this notebook together.

In [21]:
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

joblib.dump(selected_model, "models/creditwise_selected_pipeline.joblib")
validation_results.to_csv("outputs/validation_model_comparison.csv", index=False)
final_results.to_csv("outputs/test_model_comparison.csv", index=False)
importance.to_csv("outputs/permutation_feature_importance.csv", index=False)
group_metrics.to_csv("outputs/gender_subgroup_diagnostics.csv", index=False)

print("Saved selected pipeline and evaluation tables.")

Saved selected pipeline and evaluation tables.


## 15. Conclusions and limitations (complete after running)

Use the actual results generated above to answer:
1. Which model was selected on validation F1, and why?
2. How did it compare with the Dummy baseline and other classifiers on the test set?
3. What do precision, recall, F1 and ROC-AUC reveal that accuracy alone does not?
4. Which errors were most common, and what could their business consequences be?
5. Which input features were most influential according to permutation importance?
6. What did the subgroup diagnostic show, and what can it not prove?

### Limitations to state
- The provided dataset has 1,000 rows, with 50 rows missing the approval target; only labelled rows can train and evaluate a supervised classifier.
- The supplied problem statement does not identify an independently verifiable original data source or collection methodology. Do not describe the dataset as representative of all Indian borrowers without evidence.
- Historical approval labels may encode past human decisions and their biases. Predicting those labels is not the same as determining creditworthiness or making a fair lending decision.
- Model outputs should support human review, not automatically approve or reject real loan applications.
- Results from this dataset require external validation before any real-world deployment.